FAST — NUCES, Karachi Campus

<div style="border-bottom:3px solid #1F3864;padding-bottom:10px">
<h1 style="color:#1F3864;margin-bottom:2px">Lab 05 — Object-Oriented Programming</h1>
<b>Programming for Artificial Intelligence</b><br>

Department of Artificial Intelligence &amp; Data Science<br>

<i>Prepared by Khadeejah Ashraf, Lab Instructor</i>
</div>

Read the explanation, run the code cell below it with **Shift + Enter**, and compare what you
get with what you expected. Cells marked **Your turn** are for you to fill in.

**Contents**

1. From functions to objects  
2. Classes and objects  
3. The constructor and instance attributes  
4. Methods and `self`  
5. Printing an object — `__str__` and friends  
6. Encapsulation  
7. Inheritance  
8. Method overriding and `super()`  
9. Polymorphism  
10. Abstract base classes  
11. Class Tasks (C1 – C5)  
12. Home Tasks


---
## 1. From Functions to Objects

In Lab 04 the data lived in a dictionary and the functions lived elsewhere. Nothing tied them
together, so a typo in a key went unnoticed.

In [ ]:
# procedural: the data and the code that understands it are separate
student = {'name': 'Ali', 'marks': [78, 45, 90]}

def average(record):
    return round(sum(record['marks']) / len(record['marks']), 2)

print(average(student))
student['mrks'] = []        # a typo nothing will catch until much later
print(student)

71.0
{'name': 'Ali', 'marks': [78, 45, 90], 'mrks': []}


In [ ]:
# object-oriented: one unit, and every object is built the same way
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):
        return round(sum(self.marks) / len(self.marks), 2)

ali = Student('Ali', [78, 45, 90])
print(ali.name, ali.average())      # the object already knows its own marks

Ali 71.0


---
## 2. Classes and Objects

A **class** is the blueprint; an **object** (instance) is one thing built from it. Calling the
class creates an object.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

ali  = Student('Ali', [78, 45, 90])
sara = Student('Sara', [88, 92, 95])

print(ali.name, sara.name)
print(type(ali))
print(isinstance(ali, Student))
print(ali is sara)          # two separate objects

Ali Sara
<class '__main__.Student'>
True
False


### 2.1 Class attributes vs instance attributes

An attribute in the class body is shared by every object. An attribute set through `self`
belongs to one object only.

In [ ]:
class Student:
    school = 'FAST NUCES'          # class attribute - shared
    count = 0                      # shared counter

    def __init__(self, name):
        self.name = name           # instance attribute - one per object
        Student.count += 1

ali  = Student('Ali')
sara = Student('Sara')

print(ali.school, sara.school)
print(Student.count)
print(ali.name, sara.name)

FAST NUCES FAST NUCES
2
Ali Sara


In [ ]:
# a trap: assigning through the OBJECT hides the class attribute, it does not change it
ali.school = 'Somewhere else'
print(ali.school, sara.school, Student.school)

Student.school = 'FAST NUCES Karachi'    # this is how you change the shared value
print(sara.school, Student.school)

Somewhere else FAST NUCES FAST NUCES
FAST NUCES Karachi FAST NUCES Karachi


**Your turn.** Write a class `Book` with `title`, `author` and `pages` set in the constructor,
create two books, and print each one's title and page count.

In [ ]:
# Your code here

---
## 3. The Constructor and Instance Attributes

`__init__` runs automatically on every new object. You never call it yourself, and it returns
nothing — its job is to set the object's starting attributes.

In [ ]:
class BankAccount:
    def __init__(self, holder, balance=0):
        self.holder = holder
        self.balance = balance
        self.history = []          # built inside __init__, so one list per object

a = BankAccount('Ali', 5000)
b = BankAccount('Sara')            # balance takes its default

print(a.holder, a.balance)
print(b.holder, b.balance)
print(a.history is b.history)      # separate lists, not one shared list

Ali 5000
Sara 0
False


In [ ]:
# the constructor is the right place to reject impossible data
class BankAccount:
    def __init__(self, holder, balance=0):
        if balance < 0:
            raise ValueError("balance cannot be negative")
        self.holder = holder
        self.balance = balance

try:
    BankAccount('Ali', -100)
except ValueError as err:
    print("rejected:", err)

rejected: balance cannot be negative


---
## 4. Methods and `self`

A method's first parameter is always `self` — the object it was called on. You write it in the
definition but never pass it at the call: `ali.average()` runs `Student.average(ali)`.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):                 # reads this object's data
        return round(sum(self.marks) / len(self.marks), 2)

    def add_mark(self, mark):          # changes this object's data
        self.marks.append(mark)

    def grade(self):                   # a method may call another method
        avg = self.average()
        if avg >= 80:
            return 'A'
        elif avg >= 70:
            return 'B'
        elif avg >= 60:
            return 'C'
        return 'F'

ali = Student('Ali', [78, 45, 90])
print(ali.average(), ali.grade())

ali.add_mark(100)
print(ali.marks, ali.average(), ali.grade())

print(Student.average(ali))            # the same call written the long way

71.0 B
[78, 45, 90, 100] 78.25 B
78.25


In [ ]:
# forgetting self - run it to see the message you will meet often
class Broken:
    def hello():
        print("hi")

try:
    Broken().hello()
except TypeError as err:
    print("TypeError:", err)

TypeError: Broken.hello() takes 0 positional arguments but 1 was given


### 4.1 Class methods and static methods

In [ ]:
class Student:
    count = 0

    def __init__(self, name):
        self.name = name
        Student.count += 1

    @classmethod
    def how_many(cls):                 # works on the class, not one object
        return cls.count

    @staticmethod
    def is_passing(mark):              # a helper that needs no object at all
        return mark >= 50

Student('Ali'); Student('Sara')
print(Student.how_many())
print(Student.is_passing(45), Student.is_passing(78))

2
False True


**Your turn.** Add to your `Book` class a method `is_long()` returning `True` when the book has
more than 300 pages, and a class attribute `library` shared by every book. Test both.

In [ ]:
# Your code here

---
## 5. Printing an Object — `__str__` and Friends

Methods whose names begin and end with two underscores hook your class into the language's own
syntax.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

plain = Student('Ali', [78, 45, 90])
print(plain)                 # unhelpful without __str__

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def __str__(self):
        return "Student(" + self.name + ", " + str(len(self.marks)) + " marks)"

    def __len__(self):
        return len(self.marks)

    def __eq__(self, other):
        return self.name == other.name

ali   = Student('Ali', [78, 45, 90])
clone = Student('Ali', [10, 20])

print(ali)             # __str__
print(len(ali))        # __len__
print(ali == clone)    # __eq__

Student(Ali, 3 marks)
3
True


---
## 6. Encapsulation

| Written as | Meaning |
|---|---|
| `self.balance` | Public — anyone may read and change it |
| `self._bank` | Protected by convention — internal, but subclasses may use it |
| `self.__balance` | Private — the name is mangled to `_ClassName__balance` |

In [ ]:
class BankAccount:
    def __init__(self, holder, balance=0):
        self.holder = holder        # public
        self._bank = 'HBL'          # protected by convention
        self.__balance = balance    # private

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.__balance += amount

    def get_balance(self):
        return self.__balance

a = BankAccount('Ali', 5000)
a.deposit(1500)
print(a.holder, a.get_balance())
print(a._bank)                      # possible, but you are trespassing

try:
    print(a.__balance)
except AttributeError as err:
    print("AttributeError:", err)

Ali 6500
HBL
AttributeError: 'BankAccount' object has no attribute '__balance'


In [ ]:
# name mangling is protection against accidents, not security
print(a._BankAccount__balance)
print([n for n in vars(a)])

6500
['holder', '_bank', '_BankAccount__balance']


### 6.1 `@property` — attribute syntax with validation behind it

In [ ]:
class Student:
    def __init__(self, name, marks=0):
        self.name = name
        self.__marks = marks

    @property
    def marks(self):                 # reading obj.marks runs this
        return self.__marks

    @marks.setter
    def marks(self, value):          # assigning obj.marks = x runs this
        if not 0 <= value <= 100:
            raise ValueError("marks must be between 0 and 100")
        self.__marks = value

s = Student('Ali')
s.marks = 78
print(s.marks)

try:
    s.marks = 150
except ValueError as err:
    print("rejected:", err)

78
rejected: marks must be between 0 and 100


**Your turn.** Give your `Book` class a private `__price` attribute, a `price` property that
reads it, and a setter that refuses a negative price.

In [ ]:
# Your code here

---
## 7. Inheritance

A child class automatically gets every attribute and method of its parent and changes only what
differs. Look for the "is a" relationship: a `Bus` **is a** `Vehicle`.

In [ ]:
class Vehicle:
    def __init__(self, make, seats):
        self.make = make
        self.seats = seats

    def fare(self):
        return self.seats * 100

    def describe(self):
        return self.make + " with " + str(self.seats) + " seats"


class Bus(Vehicle):        # inherits everything
    pass                   # ...and adds nothing yet


b = Bus('Hino', 50)
print(b.describe())
print(b.fare())
print(isinstance(b, Bus), isinstance(b, Vehicle), issubclass(Bus, Vehicle))

Hino with 50 seats
5000
True True True


### 7.1 Extending the parent with `super()`

The child takes the extra values, hands the shared ones up to the parent, then sets what is new.

In [ ]:
class Bus(Vehicle):
    def __init__(self, make, seats, route):
        super().__init__(make, seats)     # let Vehicle set make and seats
        self.route = route                # then add what is new

    def describe(self):
        return super().describe() + " on route " + self.route


b = Bus('Hino', 50, 'Karachi-Hyderabad')
print(b.make, b.seats, b.route)
print(b.describe())

Hino 50 Karachi-Hyderabad
Hino with 50 seats on route Karachi-Hyderabad


### 7.2 Multilevel inheritance and the method resolution order

In [ ]:
class Machine:
    def __init__(self, make):
        self.make = make

class Car(Machine):
    def __init__(self, make, doors):
        super().__init__(make)
        self.doors = doors

class SportsCar(Car):
    def __init__(self, make, doors, top_speed):
        super().__init__(make, doors)
        self.top_speed = top_speed

s = SportsCar('Toyota', 2, 250)
print(s.make, s.doors, s.top_speed)
print([c.__name__ for c in SportsCar.__mro__])

Toyota 2 250
['SportsCar', 'Car', 'Machine', 'object']


In [ ]:
# multiple inheritance: the parents are searched LEFT TO RIGHT
class Engine:
    def start(self):
        return "engine started"

class Radio:
    def start(self):
        return "radio started"

class Hybrid(Engine, Radio):
    pass

class Hybrid2(Radio, Engine):     # the same parents, the other way round
    pass

print(Hybrid().start())
print(Hybrid2().start())
print([c.__name__ for c in Hybrid.__mro__])

engine started
radio started
['Hybrid', 'Engine', 'Radio', 'object']


---
## 8. Method Overriding and `super()`

The search for a method starts at the child, so a child's version wins over the parent's.

In [ ]:
class Vehicle:
    def __init__(self, make, seats):
        self.make, self.seats = make, seats

    def fare(self):
        return self.seats * 100


class Bus(Vehicle):
    def fare(self):                       # overrides Vehicle.fare completely
        return self.seats * 100 * 1.10


class Rickshaw(Vehicle):
    def fare(self):
        return self.seats * 60


for v in [Vehicle('Generic', 4), Bus('Hino', 50), Rickshaw('Sazgar', 3)]:
    print(type(v).__name__, v.fare())

Vehicle 400
Bus 5500.0
Rickshaw 180


In [ ]:
# an override can EXTEND rather than replace, by calling super()
class Bus(Vehicle):
    def fare(self):
        base = super().fare()             # reuse Vehicle's calculation
        return base + base * 0.10         # then add the bus-specific charge

print(Bus('Hino', 50).fare())

5500.0


**Your turn.** Derive `SchoolBus` from `Bus` with a fixed fare of `2500` regardless of the
number of seats, create one, print its fare, and print `SchoolBus.__mro__`.

In [ ]:
# Your code here

---
## 9. Polymorphism

The same call means different things depending on the object it lands on. You have relied on
this since Lab 01.

In [ ]:
for value in ["abc", [1, 2, 3, 4], {'a': 1}]:
    print(type(value).__name__, len(value))

print(3 + 4)
print("Prog" + "ramming")

str 3
list 4
dict 1
7
Programming


In [ ]:
# duck typing: these three classes share no parent at all
class Car:
    def describe(self):
        return "A car carries a small family."

class Truck:
    def describe(self):
        return "A truck carries cargo."

class Rickshaw3:
    def describe(self):
        return "A rickshaw carries three passengers."

for vehicle in [Car(), Truck(), Rickshaw3()]:
    print(vehicle.describe())      # nothing here asks what type it is

A car carries a small family.
A truck carries cargo.
A rickshaw carries three passengers.


In [ ]:
# one function, any class that has make and fare()
def print_details(vehicle):
    """Work with any object that has a make and a fare()."""
    print(vehicle.make, "->", vehicle.fare())

print_details(Bus('Hino', 50))
print_details(Rickshaw('Sazgar', 3))

Hino -> 5500.0
Sazgar -> 180


### 9.1 Operator overloading

In [ ]:
class Money:
    def __init__(self, rupees):
        self.rupees = rupees

    def __add__(self, other):
        return Money(self.rupees + other.rupees)

    def __str__(self):
        return "Rs " + str(self.rupees)

print(Money(1500) + Money(2500))

Rs 4000


In [ ]:
# Python has no compile-time overloading: the second definition replaces the first
def product(a, b):
    return a * b

def product(a, b, c):
    return a * b * c

try:
    print(product(4, 5))
except TypeError as err:
    print("TypeError:", err)

print(product(4, 5, 5))     # only the last definition survives

TypeError: product() missing 1 required positional argument: 'c'
100


---
## 10. Abstract Base Classes

A parent that exists only to state what its children must provide. A class deriving from `ABC`
with an `@abstractmethod` cannot be instantiated, and neither can a subclass that fails to
implement it.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def area(self):
        """Every concrete shape must supply this."""

    def describe(self):                 # a concrete method, inherited as usual
        return self.name + " has area " + str(round(self.area(), 2))


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__('Rectangle')
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        super().__init__('Circle')
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


for shape in [Rectangle(4, 5), Circle(3)]:
    print(shape.describe())

Rectangle has area 20
Circle has area 28.27


In [ ]:
# the abstract class itself cannot be instantiated
try:
    Shape('generic')
except TypeError as err:
    print("TypeError:", err)

# nor can a subclass that forgets to implement area()
class Hexagon(Shape):
    def __init__(self, side):
        super().__init__('Hexagon')
        self.side = side

try:
    Hexagon(4)
except TypeError as err:
    print("TypeError:", err)

TypeError: Can't instantiate abstract class Shape without an implementation for abstract method 'area'
TypeError: Can't instantiate abstract class Hexagon without an implementation for abstract method 'area'


---
# 11. Class Tasks

Complete all five during the lab session and demonstrate them to the instructor before you
leave. Every class needs a docstring, every attribute must be set in the constructor, and only
a display method may call `print()`.

### C1 — A Book Class
*classes · constructors · attributes · methods*

Write a class `Book` with the class attribute `library = 'FAST NUCES Library'` and the instance
attributes `title`, `author`, `pages` and `issued` (starting as `False`). Give it:

* `issue()` — marks the book as issued and returns a message, or refuses if it is already out
* `return_book()` — reverses that
* `is_long()` — `True` when the book has more than 300 pages
* `__str__` — printing a book shows its title, author and current status

Create three books, issue two of them, try to issue one of those a second time, and print all
three. Then add a class attribute `count` recording how many books have been created, and print
it through a `@classmethod` called `how_many()`.

Finish with one line showing that changing `Book.library` changes it for every existing object,
and a comment explaining why.

In [ ]:
# C1 - A Book Class
# Name:
# Roll number:

# Your code here

### C2 — A Bank Account
*encapsulation*

`__init__` takes the holder's name, a starting balance (default `0`) and a four-digit security
code. The balance and the security code must be **private**; the account number **protected**.
Provide:

* `deposit(amount)` and `withdraw(amount)`, raising `ValueError` on a non-positive amount or an
  overdraw
* a `balance` `@property` that allows reading but not direct assignment
* `statement()` — prints the holder, the account number and the last five transactions from a
  private history list
* `verify(code)` — returns `True` or `False` without ever revealing the stored code

Demonstrate that `account.__balance` raises `AttributeError` from outside the class, then that
`account._BankAccount__balance` reaches it anyway, and explain in a comment what that tells you
about privacy in Python.

In [ ]:
# C2 - A Bank Account
# Name:
# Roll number:

# Your code here

### C3 — A Vehicle Hierarchy
*inheritance · overriding · super()*

Base class `Vehicle` with `make`, `model`, `seats`, `fare()` returning `seats * 100`, and
`describe()`. Derive:

* `Bus` — overrides `fare()` to add a 10% maintenance charge
* `Car` — adds an `ac` attribute and adds 25% when the air conditioning is on
* `Truck` — replaces `seats` with `load_tonnes` and charges `load_tonnes * 500`

Every override of `describe()` must call `super().describe()` instead of repeating it. Put four
vehicles in a list, loop over it once, and print each description and fare. Then derive
`SchoolBus` from `Bus` with a fixed monthly fare, print `SchoolBus.__mro__`, and state in a
comment which class supplies `describe()` for a `SchoolBus` object and why.

In [ ]:
# C3 - A Vehicle Hierarchy
# Name:
# Roll number:

# Your code here

### C4 — Shapes
*abstraction · polymorphism*

Using `ABC` and `@abstractmethod`, write an abstract `Shape` with abstract `area()` and
`perimeter()` and a concrete `summary()` that uses both. Implement `Rectangle`, `Square`
(derived from `Rectangle`, **not** from `Shape`), `Triangle` (three sides, Heron's formula) and
`Circle`. Store one of each in a list, print every summary with a single loop, then print the
shapes sorted from the largest area to the smallest using `sorted()` with a `key`.

Show — inside `try` / `except` so the notebook still runs to the end — that `Shape()` raises a
`TypeError`, and that a subclass which forgets `perimeter()` cannot be instantiated either.

In [ ]:
# C4 - Shapes
# Name:
# Roll number:

# Your code here

### C5 — A Small Library System
*the whole lab in one program*

Three classes:

* `LibraryItem` — abstract base with `title`, a private `__item_id`, an `is_available` flag, an
  abstract `loan_period_days()` and a concrete `describe()`
* `PrintedBook` and `DVD` — derived, with different loan periods and their own extra attributes
* `Member` — a name, a private contact number and a list of borrowed items, with `borrow(item)`
  and `give_back(item)` that update both the member and the item and refuse to borrow something
  already out, plus a `history()` method

Finally write one function `show(entity)` that prints the details of **any** of these objects —
book, DVD or member — to demonstrate polymorphism. Your driver code must create at least two
members and four items, perform a borrow, a refused borrow and a return, and end by printing
every member's history.

In [ ]:
# C5 - A Small Library System
# Name:
# Roll number:

# Your code here

---
# 12. Home Tasks

1. Class `Rectangle` with `length` and `width`, methods `area()` and `perimeter()`, and a `__str__`. Create three and print each with its area and perimeter.
2. Class `Student` with `roll`, `name` and three marks; methods `total()`, `average()`, `grade()`, `display()`. Create four and print the topper's name.
3. Class `Circle` whose radius is private and exposed through a `@property`; the setter rejects a negative radius with `ValueError`. Add `area()` and `circumference()` and demonstrate a valid and an invalid assignment.
4. Create a `Bus` class inheriting from `Vehicle`. The default fare is `seating_capacity * 100`; for a bus add a further 10% as a maintenance charge. Print the fare of a car and of a bus.
5. Abstract base class `Shape` with abstract `area()`. Derive `Rectangle`, `Triangle` and `Square`, implement `area()` in each, and print all three areas from one loop.
6. Class `Account` with private `account_no`, `account_bal` and `security_code`. Initialise them through a public method (not the constructor) and print all the data through another public method. Explain in a comment why they cannot be printed directly.
7. Class `Student` displays id and name; `Marks` (derived from `Student`) displays marks for algorithms, data science and calculus; `Result` (derived from `Marks`) calculates and displays the total and average. In the driver code create one `Result` object and display everything.
8. Class `Employee` with `name`, `salary` and `calculate_bonus()` returning 10%. Derive `Manager` (20%, plus a `hire()` method) and `Developer` (plus a `write_code()` method), then `SeniorManager` from `Manager` (30%). Print every employee's name, role and bonus from one loop.
9. Class `Time` storing hours and minutes, with `__str__` printing `hh:mm` and `__add__` in which 75 minutes correctly becomes one hour fifteen. Add three times together.
10. Class `Playlist` holding song titles, with `add`, `remove`, `__len__` and a numbered `__str__`. Removing a song that is not there raises `ValueError`.
11. Multiple inheritance: `Engine` and `MusicSystem` both with a `start()` method, and `Car` inheriting from both. Print `Car().start()` and `Car.__mro__`, then swap the parent order and explain the change in a comment.
12. A vehicle-rental system: a hierarchy of `Car`, `SUV` and `Truck` with make, model, price per day and availability; methods to check availability, cost a rental of *n* days, and display details, with availability changing on rent and return. Add a `Customer` class (name, private contact number, rented vehicles, rental history) and a `RentalReservation` class linking a customer to a vehicle with start and end dates. Write one function that can display any vehicle or any reservation.

---
<div style="text-align:center;color:#666"><i>End of Lab 05 — FAST NUCES, Karachi Campus</i></div>

### `You must complete all Class and Home Tasks in the same .ipynb file, and submit using the naming format Lab 05 - Object Oriented Programming_<your_Roll_Number>.ipynb.`